### **Purpose:** load all raw CSVs, standardize date column, merge by date, and create train/test tables.

In [1]:
import pandas as pd
import numpy as np 
import warnings
import os
import glob
from pathlib import Path

In [2]:
pd.set_option('display.max_columns', None)  # Show ALL columns, don't hide any
pd.set_option('display.max_rows', 100)     # Show up to 100 rows
pd.set_option('display.precision', 2)

In [3]:
warnings.filterwarnings('ignore')

In [4]:
train_dir = Path("../data/raw/train")
test_dir = Path("../data/raw/test")

In [5]:
train_files = sorted(train_dir.glob("*.csv"))
test_files = sorted(test_dir.glob("*.csv"))

In [6]:
print("Train files:", [f.name for f in train_files])
print("Test files:", [f.name for f in train_files])

Train files: ['5cm soil moist CFS_train.csv', 'dew point_train.csv', 'max surface temp era5 max_train.csv', 'sea level pressure era5_train.csv', 'solution_train.csv', 'sst_cameroon_train.csv', 'sst_indian_ocean_train.csv', 'sudd water deficit_train.csv', 'surface pressure era5_train.csv', 'temp_trimmed_train.csv', 'u10_2002_2019_JulOct_train.csv', 'vapour pressure deficit_train.csv']
Test files: ['5cm soil moist CFS_train.csv', 'dew point_train.csv', 'max surface temp era5 max_train.csv', 'sea level pressure era5_train.csv', 'solution_train.csv', 'sst_cameroon_train.csv', 'sst_indian_ocean_train.csv', 'sudd water deficit_train.csv', 'surface pressure era5_train.csv', 'temp_trimmed_train.csv', 'u10_2002_2019_JulOct_train.csv', 'vapour pressure deficit_train.csv']


In [10]:
def _find_date_col(columns):
    # normalize: lower + strip
    cols = [c.strip().lower() for c in columns]
    # common date column names 
    candidates = ["dates", "datetime", "time", "day"]
    for cand in cols:
        return columns[cols.index(cand)]
    # fallback: first column if looks like date 
    return columns[0]

In [11]:
def load_csv(path):
    df = pd.read_csv(path)
    date_col = _find_date_col(df.columns)
    df= df.rename(columns={date_col: "date"})
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    return df

In [13]:
# Smoke test on one file if availabe
if train_files: 
    sample_df = load_csv(train_files[0])
    display(sample_df.head())
    print(sample_df.dtypes)

,date,5cm_soli_moist
0,2002-07-01,0.14
1,2002-07-02,0.14
2,2002-07-03,0.14
3,2002-07-04,0.14
4,2002-07-05,0.14


date              datetime64[ns]
5cm_soli_moist           float64
dtype: object


In [14]:
train_data = {}
train_meta = []

for f in train_files:
    df = load_csv(f) 
    train_data[f.stem] = df
    train_meta.append({
        "file": f.name,
        "rows": df.shape[0],
        "cols":df.shape[1],
        "date_nulls": int(df["date"].isna().sum())
    })

In [15]:
train_meta_df = pd.DataFrame(train_meta).sort_values("file")
display(train_meta_df)

,file,rows,cols,date_nulls
0,5cm soil moist CFS_train.csv,2214,2,0
1,dew point_train.csv,2214,2,0
2,max surface temp era5 max_train.csv,2214,2,0
3,sea level pressure era5_train.csv,2214,2,0
4,solution_train.csv,2214,2,0
5,sst_cameroon_train.csv,2214,5,0
6,sst_indian_ocean_train.csv,2214,5,0
7,sudd water deficit_train.csv,2214,2,0
8,surface pressure era5_train.csv,2214,2,0
9,temp_trimmed_train.csv,2214,2,0


In [18]:
test_data = {}
test_meta = []

for f_test in test_files:
    df = load_csv(f_test)
    test_data[f_test.stem] = df
    test_meta.append({
        "file": f.name,
        "rows": df.shape[0],
        "cols": df.shape[1],
        "date_nulls": int(df["date"].isna().sum())
    })

In [19]:
def merge_on_date(data_dict):
    dfs = list(data_dict.values())
    merged = dfs[0]
    for df in dfs[1:]:
        merged = merged.merge(df, on="date", how="left")
    return merged

In [20]:
train_merged = merge_on_date(train_data)

In [21]:
print("Train merged shape:", train_merged.shape)
display(train_merged.head())
print("Train colmuns:", list(train_merged.columns))

Train merged shape: (259038, 21)


,date,5cm_soli_moist,mean_dew_point_temp,max_temp,sea_level_pressure,dryspell_warn_7d,mean temperature kelvin_x,mean temperature deg C_x,mean temperature uncertainty_x,fraction of sea-ice-covered ocean_x,mean temperature kelvin_y,mean temperature deg C_y,mean temperature uncertainty_y,fraction of sea-ice-covered ocean_y,potential_water_deficit,surface_pressure,2m_temp,latitude,longitude,u10,vapor_pressure_dificit
0,2002-07-01,0.14,16.31,40.13,101.01,0,299.21,26.06,0.19,0.0,301.99,28.84,0.12,0.0,-4.61,95.0,32.73,12.5,33.50,0.58,5.45
1,2002-07-01,0.14,16.31,40.13,101.01,0,299.21,26.06,0.19,0.0,301.99,28.84,0.12,0.0,-4.61,95.0,32.73,12.5,33.75,0.65,5.45
2,2002-07-01,0.14,16.31,40.13,101.01,0,299.21,26.06,0.19,0.0,301.99,28.84,0.12,0.0,-4.61,95.0,32.73,12.5,34.00,0.63,5.45
3,2002-07-01,0.14,16.31,40.13,101.01,0,299.21,26.06,0.19,0.0,301.99,28.84,0.12,0.0,-4.61,95.0,32.73,12.5,34.25,0.59,5.45
4,2002-07-01,0.14,16.31,40.13,101.01,0,299.21,26.06,0.19,0.0,301.99,28.84,0.12,0.0,-4.61,95.0,32.73,12.5,34.50,0.79,5.45


Train colmuns: ['date', '5cm_soli_moist', 'mean_dew_point_temp', 'max_temp', 'sea_level_pressure', 'dryspell_warn_7d', 'mean temperature kelvin_x', 'mean temperature deg C_x', 'mean temperature uncertainty_x', 'fraction of sea-ice-covered ocean_x', 'mean temperature kelvin_y', 'mean temperature deg C_y', 'mean temperature uncertainty_y', 'fraction of sea-ice-covered ocean_y', 'potential_water_deficit', 'surface_pressure', '2m_temp', 'latitude', 'longitude', 'u10', 'vapor_pressure_dificit']
